# LIBERO VLA / WM Full Flow Lab

这个 notebook 的目标不是只介绍 `LIBERO`，而是帮你把下面这条链真正串起来：

1. 看懂 `LIBERO` 原始 `hdf5` 格式
2. 创建 `train / val` 切分
3. 明确 `VLA` 和 `WM` 各自应该吃什么数据格式
4. 对上本机已有的训练入口
5. 输出统一的验证结果：`json + mp4 + action_trace.npz`

建议你把它当成 `LIBERO` 学习和实验总控台，而不是当成模型实现文件。

In [ ]:
from pathlib import Path
import json
import subprocess
import textwrap

REPO_ROOT = Path('/home/gjw/MyProjects/autodl_unplug_charger_transformer_fm')
LIBERO_ROOT = REPO_ROOT / 'libero'
TOOLS_ROOT = LIBERO_ROOT / 'headless_tools'
SCRIPTS_ROOT = LIBERO_ROOT / 'scripts'
MANIFEST_ROOT = LIBERO_ROOT / 'manifests'

LIBERO_DATA_ROOT = TOOLS_ROOT / 'data' / 'libero_datasets'
demo_candidates = sorted(LIBERO_DATA_ROOT.rglob('*.hdf5'))
DEFAULT_DEMO_FILE = demo_candidates[0] if demo_candidates else LIBERO_DATA_ROOT / 'libero_spatial' / 'your_task_demo.hdf5'

print('REPO_ROOT =', REPO_ROOT)
print('LIBERO_ROOT =', LIBERO_ROOT)
print('TOOLS_ROOT =', TOOLS_ROOT)
print('SCRIPTS_ROOT =', SCRIPTS_ROOT)
print('DEFAULT_DEMO_FILE =', DEFAULT_DEMO_FILE)
print('demo_exists =', DEFAULT_DEMO_FILE.exists())
print('num_demo_candidates =', len(demo_candidates))

## 0. 先建立正确的数据分层

你之后所有工作都尽量不要直接把“原始 `LIBERO hdf5`”塞进训练器，而是分成这三层：

- `raw LIBERO hdf5`：benchmark 官方原始演示
- `VLA training format`：更适合 step-level 的 `image + language + state + action`
- `WM training format`：更适合 sequence / latent-level 的 `video_latent + action_chunk + language`

对于你自己的研究路线，可以这样映射：

- `多模态融合`：重点落在 `VLA` 输入建模
- `因果特征解耦`：可落在 `VLA token` 或 `WM latent`
- `域适应迁移`：主要看 train / val / OOD split 设计
- `因果可解释性`：主要看 `counterfactual rollout` 和 `activation / latent intervention`

In [ ]:
data_layers = {
    'raw_libero_hdf5': {
        'best_for': ['看 episode 结构', '看 action 维度', '看 obs 命名', '做 train/val manifest'],
        'typical_fields': ['data/demo_x/states', 'data/demo_x/actions', 'data/demo_x/obs/...'],
    },
    'vla_format': {
        'best_for': ['policy 学习', '多模态融合', 'language-conditioned action'],
        'typical_fields': ['images', 'language_instruction', 'robot_state', 'action'],
    },
    'wm_format': {
        'best_for': ['latent dynamics', 'counterfactual rollout', 'causal disentanglement'],
        'typical_fields': ['video_latents', 'action_chunks', 'language', 'optional proprio'],
    },
}
print(json.dumps(data_layers, indent=2, ensure_ascii=False))

## 1. 验证 headless 仿真环境

先确认：

- `LIBERO` 能导入
- `MuJoCo + robosuite + EGL` 离屏渲染能跑
- 训练环境也能做验证

下面这个 cell 默认只打印命令；把 `RUN = True` 再执行，就会真的运行。

In [ ]:
RUN = False

def run_cmd(cmd: str, run: bool = False):
    print(cmd)
    if run:
        return subprocess.run(cmd, shell=True, check=True, text=True)
    return None

smoke_cmd_sim = (
    "bash -lc 'source /home/gjw/MyProjects/autodl_unplug_charger_transformer_fm/libero/headless_tools/activate_libero_5090.sh "
    "&& python /home/gjw/MyProjects/autodl_unplug_charger_transformer_fm/libero/headless_tools/validate_libero_smoke.py "
    "--suite libero_spatial --task-id 0 --steps 1'"
)

smoke_cmd_train = (
    "bash -lc 'source /home/gjw/MyProjects/autodl_unplug_charger_transformer_fm/libero/headless_tools/activate_lingbot_train.sh "
    "&& python /home/gjw/MyProjects/autodl_unplug_charger_transformer_fm/libero/headless_tools/validate_libero_smoke.py "
    "--suite libero_spatial --task-id 0 --steps 1'"
)

run_cmd(smoke_cmd_sim, run=RUN)
run_cmd(smoke_cmd_train, run=RUN)

## 2. 下载 demo 数据并检查原始 `hdf5` 结构

先把官方 demo 拉下来，然后用仓库里的辅助脚本看 `episode -> states/actions/obs` 的结构。

这一步非常关键，因为你后面要做的数据转格式，本质上都建立在这层理解之上。

In [ ]:
download_cmd = (
    "bash -lc 'source /home/gjw/MyProjects/autodl_unplug_charger_transformer_fm/libero/headless_tools/activate_libero_5090.sh "
    "&& /home/gjw/MyProjects/autodl_unplug_charger_transformer_fm/libero/headless_tools/download_libero_datasets.sh libero_spatial'"
)

inspect_cmd = (
    f"bash -lc 'source /home/gjw/MyProjects/autodl_unplug_charger_transformer_fm/libero/headless_tools/activate_libero_5090.sh "
    f"&& python /home/gjw/MyProjects/autodl_unplug_charger_transformer_fm/libero/scripts/inspect_libero_dataset.py "
    f"--demo-file {DEFAULT_DEMO_FILE} --episode 0'"
)

run_cmd(download_cmd, run=False)
run_cmd(inspect_cmd, run=RUN)

if DEFAULT_DEMO_FILE.exists():
    import h5py
    with h5py.File(DEFAULT_DEMO_FILE, 'r') as f:
        episode_keys = sorted(f['data'].keys(), key=lambda key: int(key.split('_')[-1]))
        ep0 = f['data'][episode_keys[0]]
        print('num_episodes =', len(episode_keys))
        print('episode_0 keys =', sorted(ep0.keys()))
        if 'states' in ep0:
            print('states shape =', ep0['states'].shape)
        if 'actions' in ep0:
            print('actions shape =', ep0['actions'].shape)
        if 'obs' in ep0:
            print('obs keys preview =', sorted(list(ep0['obs'].keys()))[:12])
else:
    print('当前 demo 文件还不存在，先运行 download_cmd。')

## 3. 创建 `train / val` manifest

推荐不要复制原始 `hdf5` 来切分数据，而是先生成一个 `manifest json`。

这样做有 3 个好处：

- 不重复存数据
- train / val 口径统一
- 以后切 `OOD split` 更容易

In [ ]:
manifest_path = MANIFEST_ROOT / 'libero_single_task_split.json'

split_cmd = (
    f"bash -lc 'source /home/gjw/MyProjects/autodl_unplug_charger_transformer_fm/libero/headless_tools/activate_libero_5090.sh "
    f"&& python /home/gjw/MyProjects/autodl_unplug_charger_transformer_fm/libero/scripts/create_libero_split_manifest.py "
    f"--demo-file {DEFAULT_DEMO_FILE} --train-ratio 0.9 --seed 7 --out {manifest_path}'"
)

run_cmd(split_cmd, run=RUN)

if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text())
    print(json.dumps(manifest, indent=2, ensure_ascii=False))
else:
    print('manifest 还不存在，先运行 split_cmd。')

## 4. 从 `LIBERO raw` 到 `VLA` 训练格式

### 你需要的最小 sample

一个最基础的 `VLA` 样本应该长这样：

- `images`
- `language_instruction`
- `robot_state`
- `action`

### 你这台机器上现成可用的两条路

- `OpenVLA-OFT + RLDS`
- `pi0 / SmolVLA / 自定义 VLA + LeRobot`

如果你是为了尽快做研究 demo，我建议优先：

1. 先把 `LIBERO hdf5` 转成 `LeRobot`
2. 先跑一个轻量 VLA baseline
3. 再做 `OpenVLA-OFT` 这种更有论文辨识度的版本

In [ ]:
vla_paths = {
    'openvla_rlds_builder': '/home/gjw/MyProjects/RoboTwin/policy/openvla-oft/rlds_dataset_builder',
    'openvla_finetune': '/home/gjw/MyProjects/RoboTwin/policy/openvla-oft/vla-scripts/finetune.py',
    'openvla_eval': '/home/gjw/MyProjects/RoboTwin/policy/openvla-oft/experiments/robot/libero/run_libero_eval.py',
    'lerobot_converter_example': '/home/gjw/MyProjects/RoboTwin/policy/pi0/examples/libero/convert_libero_data_to_lerobot.py',
}
print(json.dumps(vla_paths, indent=2, ensure_ascii=False))

openvla_train_cmd = textwrap.dedent(
    """
    cd /home/gjw/MyProjects/RoboTwin/policy/openvla-oft
    torchrun --standalone --nnodes 1 --nproc-per-node 2 vla-scripts/finetune.py \
      --vla_path openvla/openvla-7b \
      --data_root_dir /path/to/tensorflow_datasets \
      --dataset_name your_libero_rlds_dataset \
      --run_root_dir /path/to/run_dir \
      --use_l1_regression True \
      --use_diffusion False \
      --use_film True \
      --num_images_in_input 2 \
      --use_proprio True \
      --batch_size 2 \
      --grad_accumulation_steps 1 \
      --learning_rate 5e-4 \
      --max_steps 100005 \
      --use_val_set True \
      --val_freq 1000 \
      --save_freq 5000 \
      --image_aug True \
      --wandb_entity your_entity \
      --wandb_project your_project \
      --run_id_note libero_single_task
    """
).strip()

openvla_eval_cmd = textwrap.dedent(
    """
    cd /home/gjw/MyProjects/RoboTwin/policy/openvla-oft/experiments/robot/libero
    python run_libero_eval.py \
      --pretrained_checkpoint /path/to/checkpoint \
      --task_suite_name libero_spatial \
      --num_trials_per_task 10 \
      --save_rollout_video True \
      --save_action_trace True
    """
).strip()

print(openvla_train_cmd)
print('\n' + '=' * 80 + '\n')
print(openvla_eval_cmd)

## 5. 从 `LIBERO raw` 到 `WM` 训练格式

`WM` 这条线里，建议你先遵循 `LingBot-VA` 已有的训练口径，因为它已经把：

- `LeRobot` 数据组织
- `video latents`
- `LIBERO` 推理 client

都准备好了。

### 你要得到的目标目录结构

参考本机已有的 `robotwin-clean-and-aug-lerobot`，一个适合 `LingBot-VA` 的任务目录大致长这样：

- `meta/episodes.jsonl`
- `meta/info.json`
- `data/chunk-000/episode_000000.parquet`
- `videos/chunk-000/observation.images.agentview/episode_000000.mp4`
- `latents/chunk-000/observation.images.agentview/episode_000000_0_XXX.pth`

也就是说：`WM` 训练不是直接吃 `LIBERO hdf5`，而是吃 `LeRobot + latents`。

In [ ]:
wm_paths = {
    'lingbot_train_config': '/home/gjw/MyProjects/lingbot-va/wan_va/configs/va_libero_train_cfg.py',
    'lingbot_train_entry': '/home/gjw/MyProjects/lingbot-va/wan_va/train.py',
    'lingbot_train_launcher': '/home/gjw/MyProjects/lingbot-va/script/run_va_posttrain.sh',
    'lingbot_eval_client': '/home/gjw/MyProjects/lingbot-va/evaluation/libero/client.py',
}
print(json.dumps(wm_paths, indent=2, ensure_ascii=False))

lingbot_train_cmd = textwrap.dedent(
    """
    cd /home/gjw/MyProjects/lingbot-va
    NGPU=1 \
    CONFIG_NAME=libero_train \
    ENABLE_WANDB=0 \
    DATASET_PATH=/path/to/libero_lerobot_dataset_root \
    EMPTY_EMB_PATH=/path/to/libero_lerobot_dataset_root/empty_emb.pt \
    NUM_STEPS=2000 \
    SAVE_ROOT=/path/to/lingbot_libero_run \
    bash script/run_va_posttrain.sh
    """
).strip()

lingbot_eval_cmd = textwrap.dedent(
    """
    bash -lc 'source /home/gjw/MyProjects/autodl_unplug_charger_transformer_fm/libero/headless_tools/activate_lingbot_train.sh \
      && cd /home/gjw/MyProjects/lingbot-va \
      && python evaluation/libero/client.py \
      --libero-benchmark libero_spatial \
      --port 29056 \
      --test-num 2 \
      --task-range 0 1 \
      --artifact-dir /home/gjw/MyProjects/autodl_unplug_charger_transformer_fm/libero/headless_tools/output/evals/lingbot_va \
      --fps 20'
    """
).strip()

print(lingbot_train_cmd)
print('\n' + '=' * 80 + '\n')
print(lingbot_eval_cmd)

## 6. 怎么把你自己的方法接进来

对你当前的研究主线，我建议你不要一开始就改官方训练器，而是先把自己的方法约束成两个标准接口：

### 你的 VLA 接口

- 输入：`images + language + robot_state`
- 输出：`action`

### 你的 WM 接口

- 输入：`video_latent_chunk + action_chunk + language`
- 输出：`future_latent / action / rollout`

这样你就能很自然地把：

- `多模态融合` 挂在输入编码器
- `因果解耦` 挂在 latent / token 结构
- `域适应迁移` 挂在 split 和 adapter
- `因果可解释性` 挂在 intervention 和 rollout 验证

而不是一开始就被某个开源仓库的细节绑死。

In [ ]:
sample_specs = {
    'custom_vla_sample': {
        'images': ['agentview', 'wrist(optional)'],
        'language_instruction': 'str',
        'robot_state': '[eef_pos, eef_rot, gripper]',
        'action': '[dx, dy, dz, droll, dpitch, dyaw, gripper]',
    },
    'custom_wm_sample': {
        'video_latents': '[T, ...]',
        'action_chunks': '[T, action_dim]',
        'language_instruction': 'str',
        'optional_proprio': '[T, proprio_dim]',
    },
}
print(json.dumps(sample_specs, indent=2, ensure_ascii=False))

## 7. 验证效果怎么输出

你现在已经有统一的 headless 输出链了，所以后面无论是 `OpenVLA-OFT`、`LingBot-VA`，还是你自己的方法，都尽量对齐到同一套产物：

- `metrics json`
- `rollout mp4`
- `action_trace.npz`

这样你后面做论文图、job talk demo、counterfactual 行为解释时，会非常顺手。

In [ ]:
replay_cmd = textwrap.dedent(
    """
    bash -lc 'source /home/gjw/MyProjects/autodl_unplug_charger_transformer_fm/libero/headless_tools/activate_libero_5090.sh \
      && python /home/gjw/MyProjects/autodl_unplug_charger_transformer_fm/libero/headless_tools/record_libero_rollout.py \
      --source action_trace \
      --suite libero_spatial \
      --task-id 0 \
      --action-file /path/to/ep000_success1.npz \
      --fps 20 \
      --include-wrist'
    """
).strip()

print(replay_cmd)

artifact_layout = {
    'openvla_eval_outputs': '/home/gjw/MyProjects/autodl_unplug_charger_transformer_fm/libero/headless_tools/output/evals/openvla_oft/...',
    'lingbot_eval_outputs': '/home/gjw/MyProjects/autodl_unplug_charger_transformer_fm/libero/headless_tools/output/evals/lingbot_va/...',
}
print(json.dumps(artifact_layout, indent=2, ensure_ascii=False))